In [2]:
import threading
import time
from queue import Queue, Empty
from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order

# -----------------------------------------------------------------------------
# TWS API Application Class
# -----------------------------------------------------------------------------
class IBApp(EClient, EWrapper):
    def __init__(self):
        EClient.__init__(self, self)
        # Shared state
        self.next_order_id = None
        self._order_id_ready = threading.Event()
        self._positions = []
        self._positions_ready = threading.Event()
        self._liquidity = None
        self._liquidity_ready = threading.Event()

    ## Called once TWS confirms connection and issues the next valid order ID
    def nextValidId(self, orderId: int):
        self.next_order_id = orderId
        self._order_id_ready.set()  # signal ready

    ## Capture positions one by one
    def position(self, account, contract, position, avgCost):
        self._positions.append({
            "account": account,
            "symbol": contract.symbol,
            "secType": contract.secType,
            "exchange": contract.exchange,
            "position": position,
            "avgCost": avgCost
        })

    ## Called once all positions have been sent
    def positionEnd(self):
        self._positions_ready.set()

    ## Capture account summary lines (we request 'ExcessLiquidity')
    def accountSummary(self, reqId, account, tag, value, currency):
        if tag == "ExcessLiquidity":
            try:
                self._liquidity = float(value)
            except ValueError:
                self._liquidity = None

    ## Called once all summary lines have been sent
    def accountSummaryEnd(self, reqId: int):
        self._liquidity_ready.set()

# -----------------------------------------------------------------------------
# Global application instance and thread management
# -----------------------------------------------------------------------------
_app = None
_app_thread = None

def connect(port: int = 7497, client_id: int = 100):
    """Connect to TWS/IB Gateway and start the message loop."""
    global _app, _app_thread
    _app = IBApp()
    _app.connect("127.0.0.1", port, clientId=client_id)
    _app_thread = threading.Thread(target=_app.run, daemon=True)
    _app_thread.start()
    # wait for nextValidId
    if not _app._order_id_ready.wait(timeout=5):
        raise RuntimeError("Failed to receive nextValidId from TWS.")

def disconnect():
    """Disconnect from TWS/IB Gateway and stop the message loop."""
    global _app, _app_thread
    if _app:
        _app.disconnect()
    if _app_thread:
        _app_thread.join(timeout=2)
    _app = None
    _app_thread = None

# -----------------------------------------------------------------------------
# Order placement helpers
# -----------------------------------------------------------------------------
def _place_market_order(symbol: str, action: str, quantity: int, exchange: str):
    """Internal: place a market order for `quantity` shares."""
    if _app is None:
        raise RuntimeError("Not connected. Call connect() first.")
    # wait for valid order ID
    order_id = _app.next_order_id
    if order_id is None:
        raise RuntimeError("No order ID available.")
    # Build contract
    contract = Contract()
    contract.symbol = symbol
    contract.secType = "STK"
    contract.exchange = exchange or "SMART"
    contract.currency = "USD"
    # Build order
    order = Order()
    order.orderId = order_id
    order.action = action.upper()
    order.orderType = "MKT"
    order.totalQuantity = quantity
    # Place
    _app.placeOrder(order_id, contract, order)
    # prepare next order ID
    _app._order_id_ready.clear()
    _app.reqIds(1)  # ask TWS for next valid ID

def buy(symbol: str, quantity: int, exchange: str = "SMART"):
    """Place a BUY market order for `quantity` shares of `symbol`."""
    _place_market_order(symbol, "BUY", quantity, exchange)

def sell(symbol: str, quantity: int, exchange: str = "SMART"):
    """Place a SELL market order for `quantity` shares of `symbol`."""
    _place_market_order(symbol, "SELL", quantity, exchange)

def close_buy(symbol: str, quantity: int, exchange: str = "SMART"):
    """Close a long position by selling `quantity` shares."""
    sell(symbol, quantity, exchange)

def close_sell(symbol: str, quantity: int, exchange: str = "SMART"):
    """Close a short position by buying back `quantity` shares."""
    buy(symbol, quantity, exchange)

# -----------------------------------------------------------------------------
# Query functions
# -----------------------------------------------------------------------------
def get_positions(timeout: float = 5.0):
    """
    Request and return a list of current open positions.
    Each entry has 'symbol', 'position', and 'avgCost'.
    """
    if _app is None:
        raise RuntimeError("Not connected. Call connect() first.")
    # clear old data
    _app._positions.clear()
    _app._positions_ready.clear()
    # ask for positions
    _app.reqPositions()
    # wait
    if not _app._positions_ready.wait(timeout=timeout):
        raise RuntimeError("Timeout waiting for positions.")
    _app.cancelPositions()
    return _app._positions.copy()

def get_excess_liquidity(timeout: float = 5.0):
    """
    Request and return 'ExcessLiquidity' from account summary (buying power).
    """
    if _app is None:
        raise RuntimeError("Not connected. Call connect() first.")
    # clear old data
    _app._liquidity = None
    _app._liquidity_ready.clear()
    # request summary for ExcessLiquidity tag
    _app.reqAccountSummary(9001, "All", "ExcessLiquidity")
    # wait
    if not _app._liquidity_ready.wait(timeout=timeout):
        raise RuntimeError("Timeout waiting for account summary.")
    _app.cancelAccountSummary(9001)
    return _app._liquidity


In [3]:
# ib_trading.py

import threading
import time
from queue import SimpleQueue

from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order

# -----------------------------------------------------------------------------
#   Low‑level IB client & wrapper
# -----------------------------------------------------------------------------
class IB(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        # will be set in nextValidId
        self._nextOrderId = None

        # for get_positions()
        self._positions = []
        self._pos_req_done = threading.Event()
        # for get_excess_liq()
        self._acct_summary = {}
        self._acct_req_done = threading.Event()

    # --- connection management -----------------------------------------------
    def connect_and_run(self, host, port, client_id):
        self.connect(host, port, client_id)
        thread = threading.Thread(target=self.run, daemon=True)
        thread.start()
        # wait for nextValidId before returning
        timeout = time.time() + 5
        while self._nextOrderId is None and time.time() < timeout:
            time.sleep(0.1)
        if self._nextOrderId is None:
            raise RuntimeError("Could not get nextValidId from TWS")
        return thread

    def disconnect_and_wait(self, thread):
        self.disconnect()
        thread.join(1)

    # --- order‑id generator -------------------------------------------------
    def nextValidId(self, orderId: int):
        self._nextOrderId = orderId
        super().nextValidId(orderId)

    # --- positions retrieval ------------------------------------------------
    def position(self, account: str, contract: Contract, position: float,
                 avgCost: float):
        self._positions.append({
            "account": account,
            "symbol": contract.symbol,
            "secType": contract.secType,
            "exchange": contract.exchange,
            "position": position,
            "avgCost": avgCost
        })

    def positionEnd(self):
        self._pos_req_done.set()

    # --- account summary retrieval ------------------------------------------
    def accountSummary(self, reqId, account, tag, value, currency):
        # grab only 'ExcessLiquidity'
        if tag == "ExcessLiquidity":
            self._acct_summary["ExcessLiquidity"] = float(value)

    def accountSummaryEnd(self, reqId: int):
        self._acct_req_done.set()

    # --- error handler (to suppress spurious warnings) ----------------------
    def error(self, reqId, errorCode, errorString):
        # only print real errors (code >= 2000)
        if errorCode >= 2000:
            print(f"[IB ERROR] reqId={reqId}, code={errorCode}, msg={errorString}")

# -----------------------------------------------------------------------------
#   Public, singleton‐style interface
# -----------------------------------------------------------------------------
_ib: IB = None
_thread: threading.Thread = None

def connect(port=7497, client_id=100, host="127.0.0.1"):
    global _ib, _thread
    if _ib is not None:
        return
    _ib = IB()
    _thread = _ib.connect_and_run(host, port, client_id)
    time.sleep(0.2)  # a little breathing room

def disconnect():
    global _ib, _thread
    if _ib is None:
        return
    _ib.disconnect_and_wait(_thread)
    _ib, _thread = None, None

def _make_contract(symbol: str, exchange: str):
    c = Contract()
    c.symbol = symbol
    c.secType = "STK"
    c.currency = "USD"
    c.exchange = exchange or "SMART"
    return c

def _place_market_order(action: str, symbol: str, quantity: float, exchange: str):
    """Internal helper: places MKT/DAY order with no extras."""
    global _ib
    oid = _ib._nextOrderId
    order = Order()
    order.orderId = oid
    order.action = action.upper()
    order.orderType = "MKT"
    order.totalQuantity = quantity
    order.tif = "DAY"
    order.eTradeOnly = False
    order.firmQuoteOnly = False
    order.outsideRth = False

    _ib.placeOrder(oid, _make_contract(symbol, exchange), order)
    _ib._nextOrderId += 1
    print(f">>> {action.upper()} {quantity}×{symbol} @MKT (orderId={oid})")

def buy(symbol: str, quantity: float, exchange: str="SMART"):
    _place_market_order("BUY", symbol, quantity, exchange)

def sell(symbol: str, quantity: float, exchange: str="SMART"):
    _place_market_order("SELL", symbol, quantity, exchange)

def close_buy(symbol: str, quantity: float, exchange: str="SMART"):
    # to close a long, you SELL the same qty
    sell(symbol, quantity, exchange)

def close_sell(symbol: str, quantity: float, exchange: str="SMART"):
    # to close a short, you BUY the same qty
    buy(symbol, quantity, exchange)

def get_positions(timeout=5.0):
    """Returns a list of current positions."""
    global _ib
    _ib._positions.clear()
    _ib._pos_req_done.clear()
    _ib.reqPositions()
    done = _ib._pos_req_done.wait(timeout)
    if not done:
        raise RuntimeError("Timed out waiting for positions")
    _ib.cancelPositions()  # clean up
    return list(_ib._positions)

def get_excess_liquidity(timeout=5.0):
    """Returns your USD excess liquidity."""
    global _ib
    _ib._acct_summary.clear()
    _ib._acct_req_done.clear()
    # reqId can be any int; we just use 0
    _ib.reqAccountSummary(0, "All", "ExcessLiquidity")
    done = _ib._acct_req_done.wait(timeout)
    _ib.cancelAccountSummary(0)
    if not done:
        raise RuntimeError("Timed out waiting for account summary")
    return _ib._acct_summary.get("ExcessLiquidity")

# -----------------------------------------------------------------------------
#   Usage example (put in your own script, *not* inside this module):
# -----------------------------------------------------------------------------
# from ib_trading import connect, buy, sell, close_buy, close_sell, get_positions, get_excess_liquidity, disconnect
#
connect(port=7497, client_id=100)
buy("AAPL", quantity=2)
sell("TSLA", quantity=1)
close_buy("AAPL", quantity=2)
close_sell("TSLA", quantity=1)
print("Positions:", get_positions())
print("Excess Liq:", get_excess_liquidity())
disconnect()


[IB ERROR] reqId=-1, code=2104, msg=Market data farm connection is OK:usfarm.nj
[IB ERROR] reqId=-1, code=2104, msg=Market data farm connection is OK:usfuture
[IB ERROR] reqId=-1, code=2104, msg=Market data farm connection is OK:eufarm
[IB ERROR] reqId=-1, code=2104, msg=Market data farm connection is OK:cashfarm
[IB ERROR] reqId=-1, code=2104, msg=Market data farm connection is OK:usfarm
[IB ERROR] reqId=-1, code=2106, msg=HMDS data farm connection is OK:euhmds
[IB ERROR] reqId=-1, code=2106, msg=HMDS data farm connection is OK:ushmds
[IB ERROR] reqId=-1, code=2158, msg=Sec-def data farm connection is OK:secdefeu
>>> BUY 2×AAPL @MKT (orderId=21)
>>> SELL 1×TSLA @MKT (orderId=22)
>>> SELL 2×AAPL @MKT (orderId=23)
>>> BUY 1×TSLA @MKT (orderId=24)
Positions: [{'account': 'DU081', 'symbol': 'TSLA', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 0.0, 'avgCost': 0.0}, {'account': 'DU081', 'symbol': 'AAPL', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 0.0, 'avgCost': 0.0}]
Excess

In [4]:
# ib_trading.py

import threading
import time
import requests

from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order

# ─────────────────────────────────────────────────────────────────────────────
#  AlphaVantage price fetch (free)
ALPHA_KEY = "F1UXC7ZGHIEUFHUN"
def get_price_from_alpha(symbol: str, interval: str="1min", output_size: str="compact") -> float:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_INTRADAY",
        "symbol": symbol,
        "interval": interval,
        "apikey": ALPHA_KEY,
        "outputsize": output_size,
    }
    r = requests.get(url, params=params, timeout=5)
    data = r.json()
    ts = data.get(f"Time Series ({interval})")
    if not ts:
        raise RuntimeError(f"No intraday data for {symbol}")
    latest = sorted(ts.keys())[-1]
    return float(ts[latest]["4. close"])

# ─────────────────────────────────────────────────────────────────────────────
class IB(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self._nextOrderId = None
        self._positions = []
        self._pos_done = threading.Event()
        self._acct = {}
        self._acct_done = threading.Event()

    # -- connection/ID --------------------------------------
    def nextValidId(self, orderId: int):
        self._nextOrderId = orderId

    def connect_and_run(self, host, port, client_id):
        self.connect(host, port, client_id)
        t = threading.Thread(target=self.run, daemon=True)
        t.start()
        # wait for nextValidId()
        end = time.time() + 5
        while self._nextOrderId is None and time.time() < end:
            time.sleep(0.1)
        if self._nextOrderId is None:
            raise RuntimeError("no nextValidId from TWS")
        return t

    def disconnect_and_wait(self, thread):
        self.disconnect()
        thread.join(1)

    # -- positions callback --------------------------------
    def position(self, account, contract: Contract, pos, avgCost):
        self._positions.append({
            "account": account,
            "symbol": contract.symbol,
            "secType": contract.secType,
            "exchange": contract.exchange,
            "position": pos,
            "avgCost": avgCost
        })
    def positionEnd(self):
        self._pos_done.set()

    # -- account summary callback --------------------------
    def accountSummary(self, reqId, account, tag, value, currency):
        if tag == "ExcessLiquidity":
            self._acct["ExcessLiquidity"] = float(value)
    def accountSummaryEnd(self, reqId):
        self._acct_done.set()

    # -- suppress only real errors -------------------------
    def error(self, reqId, code, msg):
        if code >= 2000:
            print(f"[IB ERROR] reqId={reqId} code={code} msg={msg}")

# ─────────────────────────────────────────────────────────────────────────────
_ib: IB = None
_thread: threading.Thread = None

def connect(port:int=7497, client_id:int=100, host:str="127.0.0.1"):
    global _ib, _thread
    if _ib: return
    _ib = IB()
    _thread = _ib.connect_and_run(host, port, client_id)
    time.sleep(0.2)

def disconnect():
    global _ib, _thread
    if not _ib: return
    _ib.disconnect_and_wait(_thread)
    _ib = None

def _make_contract(symbol, exchange):
    c = Contract()
    c.symbol = symbol
    c.secType = "STK"
    c.currency = "USD"
    c.exchange = exchange or "SMART"
    return c

def _place_limit_order(action, symbol, quantity, exchange):
    global _ib
    # fetch most recent price
    price = get_price_from_alpha(symbol)
    oid = _ib._nextOrderId
    order = Order()
    order.orderId       = oid
    order.action        = action.upper()
    order.orderType     = "LMT"
    order.lmtPrice      = price
    order.totalQuantity = quantity
    order.tif           = "DAY"
    # clear any weird flags
    order.eTradeOnly    = False
    order.firmQuoteOnly = False
    order.outsideRth    = False

    _ib.placeOrder(oid, _make_contract(symbol, exchange), order)
    print(f">>> {action.upper()} {quantity}×{symbol} @LMT {price:.2f} (orderId={oid})")
    _ib._nextOrderId += 1

def buy(symbol:str, quantity:float, exchange:str="SMART"):
    _place_limit_order("BUY", symbol, quantity, exchange)

def sell(symbol:str, quantity:float, exchange:str="SMART"):
    _place_limit_order("SELL", symbol, quantity, exchange)

def close_buy(symbol:str, quantity:float, exchange:str="SMART"):
    sell(symbol, quantity, exchange)

def close_sell(symbol:str, quantity:float, exchange:str="SMART"):
    buy(symbol, quantity, exchange)

def get_positions(timeout:float=5.0):
    global _ib
    _ib._positions.clear()
    _ib._pos_done.clear()
    _ib.reqPositions()
    if not _ib._pos_done.wait(timeout):
        raise RuntimeError("timeout waiting for positions")
    _ib.cancelPositions()
    return list(_ib._positions)

def get_excess_liquidity(timeout:float=5.0):
    global _ib
    _ib._acct.clear()
    _ib._acct_done.clear()
    _ib.reqAccountSummary(0, "All", "ExcessLiquidity")
    if not _ib._acct_done.wait(timeout):
        raise RuntimeError("timeout waiting for account summary")
    _ib.cancelAccountSummary(0)
    return _ib._acct.get("ExcessLiquidity", None)


In [5]:
# 1) connect once
connect(port=7497, client_id=100)

# 2) place orders as limit orders at NBBO
buy("AAPL", quantity=2)
sell("TSLA", quantity=1)

# 3) close positions
close_buy("AAPL", quantity=2)
close_sell("TSLA", quantity=1)

# 4) inspect
print("Positions:", get_positions())
print("Excess Liq:", get_excess_liquidity())

# 5) done
disconnect()

[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm.nj
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfuture
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:eufarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:cashfarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:euhmds
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:ushmds
[IB ERROR] reqId=-1 code=2158 msg=Sec-def data farm connection is OK:secdefeu
>>> BUY 2×AAPL @LMT 210.56 (orderId=25)
>>> SELL 1×TSLA @LMT 288.10 (orderId=26)
>>> SELL 2×AAPL @LMT 210.56 (orderId=27)
>>> BUY 1×TSLA @LMT 288.10 (orderId=28)
Positions: [{'account': 'DU081', 'symbol': 'TSLA', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 0.0, 'avgCost': 0.0}, {'account': 'DU081', 'symbol': 'AAPL', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 0.0, 'avgCost': 

In [6]:
import threading
import time
import requests

from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order

def _place_order(action, symbol, quantity, exchange, order_type="LMT"):
    global _ib
    price = get_price_from_alpha(symbol) if order_type == "LMT" else None
    oid = _ib._nextOrderId
    order = Order()
    order.orderId = oid
    order.action = action.upper()
    order.orderType = order_type.upper()
    if order_type.upper() == "LMT":
        order.lmtPrice = price
    order.totalQuantity = quantity
    order.tif = "DAY"
    order.eTradeOnly = False
    order.firmQuoteOnly = False
    order.outsideRth = False

    _ib.placeOrder(oid, _make_contract(symbol, exchange), order)
    msg = f"{action.upper()} {quantity}×{symbol} @{order_type.upper()}"
    msg += f" {price:.2f}" if price else ""
    print(f">>> {msg} (orderId={oid})")
    _ib._nextOrderId += 1
def buy(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    _place_order("BUY", symbol, quantity, exchange, order_type="MKT" if fast else "LMT")

def sell(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    _place_order("SELL", symbol, quantity, exchange, order_type="MKT" if fast else "LMT")

def close_buy(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    sell(symbol, quantity, exchange, fast)

def close_sell(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    buy(symbol, quantity, exchange, fast)


connect()

# Fast execution using Market Orders
buy("AAPL", 2, fast=True)
sell("TSLA", 1, fast=True)

close_buy("AAPL", 2, fast=True)
close_sell("TSLA", 1, fast=True)

print("Positions:", get_positions())
print("Excess Liq:", get_excess_liquidity())

disconnect()


[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm.nj
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfuture
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:eufarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:cashfarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:euhmds
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:ushmds
[IB ERROR] reqId=-1 code=2158 msg=Sec-def data farm connection is OK:secdefeu
>>> BUY 2×AAPL @MKT (orderId=29)
>>> SELL 1×TSLA @MKT (orderId=30)
>>> SELL 2×AAPL @MKT (orderId=31)
>>> BUY 1×TSLA @MKT (orderId=32)
Positions: [{'account': 'DU081', 'symbol': 'TSLA', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': -1.0, 'avgCost': 284.331867}, {'account': 'DU081', 'symbol': 'AAPL', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 2.0, 'avgCost': 210.230035}]
Excess 

In [7]:
connect()

# Fast execution using Market Orders
buy("AAPL", 2, fast=True)
sell("TSLA", 1, fast=True)

close_buy("AAPL", 2, fast=True)
close_sell("TSLA", 1, fast=True)

print("Positions:", get_positions())
print("Excess Liq:", get_excess_liquidity())

disconnect()


[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm.nj
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfuture
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:eufarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:cashfarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:euhmds
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:ushmds
[IB ERROR] reqId=-1 code=2158 msg=Sec-def data farm connection is OK:secdefeu
>>> BUY 2×AAPL @MKT (orderId=33)
>>> SELL 1×TSLA @MKT (orderId=34)
>>> SELL 2×AAPL @MKT (orderId=35)
>>> BUY 1×TSLA @MKT (orderId=36)
Positions: [{'account': 'DU081', 'symbol': 'TSLA', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 0.0, 'avgCost': 0.0}, {'account': 'DU081', 'symbol': 'AAPL', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 2.0, 'avgCost': 210.230035}]
Excess Liq: 999

In [8]:
import threading
import time
import requests

from ibapi.client import EClient
from ibapi.wrapper import EWrapper
from ibapi.contract import Contract
from ibapi.order import Order

# ─────────────────────────────────────────────────────────────────────────────
#  AlphaVantage price fetch (free)
ALPHA_KEY = "F1UXC7ZGHIEUFHUN"
def get_price_from_alpha(symbol: str, interval: str="1min", output_size: str="compact") -> float:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_INTRADAY",
        "symbol": symbol,
        "interval": interval,
        "apikey": ALPHA_KEY,
        "outputsize": output_size,
    }
    r = requests.get(url, params=params, timeout=5)
    data = r.json()
    ts = data.get(f"Time Series ({interval})")
    if not ts:
        raise RuntimeError(f"No intraday data for {symbol}")
    latest = sorted(ts.keys())[-1]
    return float(ts[latest]["4. close"])
# ─────────────────────────────────────────────────────────────────────────────
class IB(EWrapper, EClient):
    def __init__(self):
        EClient.__init__(self, self)
        self._nextOrderId = None
        self._positions = []
        self._pos_done = threading.Event()
        self._acct = {}
        self._acct_done = threading.Event()

    # -- connection/ID --------------------------------------
    def nextValidId(self, orderId: int):
        self._nextOrderId = orderId

    def connect_and_run(self, host, port, client_id):
        self.connect(host, port, client_id)
        t = threading.Thread(target=self.run, daemon=True)
        t.start()
        # wait for nextValidId()
        end = time.time() + 5
        while self._nextOrderId is None and time.time() < end:
            time.sleep(0.1)
        if self._nextOrderId is None:
            raise RuntimeError("no nextValidId from TWS")
        return t

    def disconnect_and_wait(self, thread):
        self.disconnect()
        thread.join(1)

    # -- positions callback --------------------------------
    def position(self, account, contract: Contract, pos, avgCost):
        self._positions.append({
            "account": account,
            "symbol": contract.symbol,
            "secType": contract.secType,
            "exchange": contract.exchange,
            "position": pos,
            "avgCost": avgCost
        })
    def positionEnd(self):
        self._pos_done.set()

    # -- account summary callback --------------------------
    def accountSummary(self, reqId, account, tag, value, currency):
        if tag == "ExcessLiquidity":
            self._acct["ExcessLiquidity"] = float(value)
    def accountSummaryEnd(self, reqId):
        self._acct_done.set()

    # -- suppress only real errors -------------------------
    def error(self, reqId, code, msg):
        if code >= 2000:
            print(f"[IB ERROR] reqId={reqId} code={code} msg={msg}")

# ─────────────────────────────────────────────────────────────────────────────
_ib: IB = None
_thread: threading.Thread = None

def connect(port:int=7497, client_id:int=100, host:str="127.0.0.1"):
    global _ib, _thread
    if _ib: return
    _ib = IB()
    _thread = _ib.connect_and_run(host, port, client_id)
    time.sleep(0.2)

def disconnect():
    global _ib, _thread
    if not _ib: return
    _ib.disconnect_and_wait(_thread)
    _ib = None
# ─────────────────────────────────────────────────────────────────────────────

def _make_contract(symbol, exchange):
    c = Contract()
    c.symbol = symbol
    c.secType = "STK"
    c.currency = "USD"
    c.exchange = exchange or "SMART"
    return c

def _place_order(action, symbol, quantity, exchange, order_type="LMT"):
    global _ib
    price = get_price_from_alpha(symbol) if order_type == "LMT" else None
    oid = _ib._nextOrderId
    order = Order()
    order.orderId = oid
    order.action = action.upper()
    order.orderType = order_type.upper()
    if order_type.upper() == "LMT":
        order.lmtPrice = price
    order.totalQuantity = quantity
    order.tif = "DAY"
    order.eTradeOnly = False
    order.firmQuoteOnly = False
    order.outsideRth = False

    _ib.placeOrder(oid, _make_contract(symbol, exchange), order)
    msg = f"{action.upper()} {quantity}×{symbol} @{order_type.upper()}"
    msg += f" {price:.2f}" if price else ""
    print(f">>> {msg} (orderId={oid})")
    _ib._nextOrderId += 1
def buy(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    _place_order("BUY", symbol, quantity, exchange, order_type="MKT" if fast else "LMT")

def sell(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    _place_order("SELL", symbol, quantity, exchange, order_type="MKT" if fast else "LMT")

def close_buy(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    sell(symbol, quantity, exchange, fast)

def close_sell(symbol: str, quantity: float, exchange: str = "SMART", fast=False):
    buy(symbol, quantity, exchange, fast)

def get_positions(timeout:float=5.0):
    global _ib
    _ib._positions.clear()
    _ib._pos_done.clear()
    _ib.reqPositions()
    if not _ib._pos_done.wait(timeout):
        raise RuntimeError("timeout waiting for positions")
    _ib.cancelPositions()
    return list(_ib._positions)

def get_excess_liquidity(timeout:float=5.0):
    global _ib
    _ib._acct.clear()
    _ib._acct_done.clear()
    _ib.reqAccountSummary(0, "All", "ExcessLiquidity")
    if not _ib._acct_done.wait(timeout):
        raise RuntimeError("timeout waiting for account summary")
    _ib.cancelAccountSummary(0)
    return _ib._acct.get("ExcessLiquidity", None)

connect()

# Fast execution using Market Orders
buy("AAPL", 2, fast=True)
sell("TSLA", 1, fast=True)

close_buy("AAPL", 2, fast=True)
close_sell("TSLA", 1, fast=True)

print("Positions:", get_positions())
print("Excess Liq:", get_excess_liquidity())

disconnect()


[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm.nj
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfuture
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:eufarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:cashfarm
[IB ERROR] reqId=-1 code=2104 msg=Market data farm connection is OK:usfarm
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:euhmds
[IB ERROR] reqId=-1 code=2106 msg=HMDS data farm connection is OK:ushmds
[IB ERROR] reqId=-1 code=2158 msg=Sec-def data farm connection is OK:secdefeu
>>> BUY 2×AAPL @MKT (orderId=37)
>>> SELL 1×TSLA @MKT (orderId=38)
>>> SELL 2×AAPL @MKT (orderId=39)
>>> BUY 1×TSLA @MKT (orderId=40)
Positions: [{'account': 'DU081', 'symbol': 'TSLA', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 0.0, 'avgCost': 0.0}, {'account': 'DU081', 'symbol': 'AAPL', 'secType': 'STK', 'exchange': 'NASDAQ', 'position': 2.0, 'avgCost': 210.230035}]
Excess Liq: 999